In [2]:
# Install required packages
%pip install -U pageindex openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


c:\Bijay\University Documents (DSU)\Research Files\Agentic AI\.venv\Scripts\python.exe: No module named pip


In [3]:
%pip install -U pageindex

Note: you may need to restart the kernel to use updated packages.


c:\Bijay\University Documents (DSU)\Research Files\Agentic AI\.venv\Scripts\python.exe: No module named pip


In [22]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()

PAGEINDEX_API_KEY = os.getenv("PAGE_INDEX_API")
GROQ_API_KEY    = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY=os.getenv("ABACUS_API_KEY")

print("PageIndex key loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing!")
print("GROQ key loaded:   ", "✅" if GROQ_API_KEY    else "❌ Missing!")
print("OPEN key loaded:   ", "✅" if OPENAI_API_KEY    else "❌ Missing!")

PageIndex key loaded: ✅
GROQ key loaded:    ✅
OPEN key loaded:    ✅


In [2]:
%pip show pageindex

Note: you may need to restart the kernel to use updated packages.


c:\Bijay\University Documents (DSU)\Research Files\Agentic AI\.venv\Scripts\python.exe: No module named pip


In [5]:
%pip install -U pageindex

Note: you may need to restart the kernel to use updated packages.


c:\Bijay\University Documents (DSU)\Research Files\Agentic AI\.venv\Scripts\python.exe: No module named pip


In [29]:
from pageindex import PageIndexClient
#from langchain_groq import ChatGroq
from groq import Groq
from langchain_openai import ChatOpenAI

pi_client     = PageIndexClient(api_key=PAGEINDEX_API_KEY)
#groq_client= ChatGroq(model="openai/gpt-oss-120b")



# Set your API key securely before running the function.
# For example, configure GROQ_API_KEY in your environment.
groq_client = Groq(
    api_key=os.environ.get("GROQ_API_KEY")
)

openai_client = ChatOpenAI(
    model="gpt-4o",
    api_key=OPENAI_API_KEY,
    base_url="https://routellm.abacus.ai/v1",
)
print("✅ PageIndex client ready")
print("✅ OpenAI client ready")

✅ PageIndex client ready
✅ OpenAI client ready


#### **Upload & Index a PDF**

What happens here:

1. Upload your PDF to the PageIndex cloud
2. PageIndex uses an LLM to read the document structure
3. Builds a hierarchical tree index (like a smart Table of Contents)
4. Returns a doc_id for all future operations

Why NO chunking?
Instead of cutting the document into arbitrary 500-token pieces, PageIndex respects the document's natural section boundaries — chapters, sub-sections, paragraphs — as the author intended.

In [9]:
# ── Upload your PDF ─────────────────────────────────────────────────────────
# Replace with the path to your PDF file
# Great candidates: Annual reports, research papers, legal docs, textbooks

PDF_PATH = "../resources/Density_Map_Guided_Object_Detection.pdf"   # ← change this

print(f"📤 Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"✅ Uploaded!")
print(f"📋 Document ID: {doc_id}")
print("   (Save this ID — you'll use it throughout the notebook)")

📤 Uploading: ../resources/Density_Map_Guided_Object_Detection.pdf
✅ Uploaded!
📋 Document ID: pi-cmveds9kv009t0ansn2bssqo1
   (Save this ID — you'll use it throughout the notebook)


In [10]:
# ── Poll until processing is complete ───────────────────────────────────────
# PageIndex builds the tree asynchronously.
# For a 50-page PDF this typically takes 30–90 seconds.

print("⏳ Building tree index...")
print("   (This runs once per document — the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")
    
    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        print("\n❌ Processing failed. Check your PDF format.")
        break
    
    time.sleep(5)

⏳ Building tree index...
   (This runs once per document — the index is cached for reuse)
   Status: completed

✅ Tree index ready!


🔍 Section 3: Inspect the Tree Structure
What the tree looks like:

Document
├── Introduction (pages 1-3)
│   └── Background (pages 1-2)
├── Financial Stability (pages 21-31)
│   ├── Monitoring Vulnerabilities (pages 22-28)
│   └── International Cooperation (pages 28-31)
└── Conclusion (pages 45-47)
Each node has:

node_id — unique ID used during retrieval
title — section heading
page_index — page number in original PDF
text — section summary (when node_summary=True)
nodes — child sections (nested)
This structure is what the LLM reasons over during retrieval.

In [11]:
# ── Fetch the full tree ─────────────────────────────────────────────────────
tree_result  = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

📊 Top-level sections: 1

🌲 Raw tree (first node):
{
  "title": "Density Map Guided Object Detection in Aerial Images",
  "node_id": "0000",
  "start_index": 1,
  "end_index": 10,
  "summary": "This section presents the research paper on DMNet (Density-Map guided object detection Network), detailing its authors, affiliations, and an abstract that introduces its application to aerial images. It covers the introduction of the framework to solve scale variance and non-uniform distribution challenges, a review of related work in general and aerial object detection alongside density estimation, and the comprehensive architecture of DMNet comprising density map generation, image cropping, and fusion detection modules. Furthermore, it outlines the experimental setup, datasets (VisDrone and UAVDT), evaluation metrics, quantitative results, and ablation studies, concluding with a summary of contributions, funding acknowledgements, and academic references.",
  "text": "",
  "nodes": [
    {
     

In [12]:
# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] Density Map Guided Object Detection in Aerial Images  (p.?)
  └─ [0001] Density Map Guided Object Detection in Aerial Images (intro)  (p.?)
  └─ [0002] Abstract  (p.?)
  └─ [0003] 1. Introduction  (p.?)
  └─ [0004] 2. Related work  (p.?)
  └─ [0005] 3. Density Map guided detection Network (DMNet)  (p.?)
    └─ [0006] 3.1. Overview  (p.?)
    └─ [0007] 3.2. Density map generation  (p.?)
    └─ [0008] 3.3. Image cropping based on density map  (p.?)
    └─ [0009] 3.4. Object detection on density crops  (p.?)
  └─ [0010] 4. Experiments  (p.?)
    └─ [0011] 4.1. Implementation details  (p.?)
    └─ [0012] 4.2. Datasets  (p.?)
    └─ [0013] 4.3. Evaluation metric  (p.?)
    └─ [0014] 4.4. Quantitative result  (p.?)
    └─ [0015] 4.5. Ablation study  (p.?)
  └─ [0016] 5. Conclusion  (p.?)
  └─ [0017] 6. Acknowledgements  (p.?)
  └─ [0018] References  (p.?)


In [13]:
# ── Count total nodes ────────────────────────────────────────────────────────
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

🔢 Total nodes in tree: 19
   Each node = one retrievable section of the document


🧠 Section 4: LLM Tree Search — The Core of PageIndex
This is where PageIndex fundamentally differs from vector RAG.

Vector RAG retrieval:
query → embed → cosine_similarity(query_vec, all_chunk_vecs) → top-k chunks
Problem: finds what's similar, not what's relevant

PageIndex retrieval:
query + tree → LLM reasons → "node 0007 and 0008 contain the answer"
Advantage: LLM understands document structure, context, and intent

The LLM acts like a human expert scanning a Table of Contents.

In [32]:
# ── LLM Tree Search Function ─────────────────────────────────────────────────

def llm_tree_search(query: str, tree: list, model: str = "openai/gpt-oss-120b") -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """
    
    # Compress tree to save tokens — only send titles + short summaries
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]  # first 150 chars
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    
    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    
    return json.loads(response.choices[0].message.content)

In [34]:
# ── Test with a sample query ─────────────────────────────────────────────────
query = "How is Density map generated?"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: How is Density map generated?

🧠 LLM Reasoning:
The query asks for the method of generating a density map. In the document hierarchy, the section titled '3.2. Density map generation' (node 0007) is the most directly relevant. The preceding overview (node 0006) describes the overall components, including the density map generation module, and could provide contextual details. Additionally, the implementation details (node 0011) mention the specific network (MCNN) used for density map generation, offering technical insight. Therefore, the nodes most likely to contain the answer are 0007, with 0006 and 0011 as supporting sections.

🎯 Selected Node IDs: ['0007', '0006', '0011']


⚙️ Section 5: Full End-to-End RAG Pipeline
3 steps:

Tree Search → LLM picks relevant node_ids
Retrieve → Fetch the actual section content from those nodes
Generate → LLM writes a grounded answer with page citations
What makes this better than vector RAG:

Retrieved content has titles + page numbers (traceable)
LLM can cite exactly which section the answer comes from
No hallucination from irrelevant chunks

In [35]:
# ── Helper: Find nodes by ID ─────────────────────────────────────────────────

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [43]:
# ── Generate answer from retrieved nodes ─────────────────────────────────────

def generate_answer(query: str, nodes: list, model: str = "openai/gpt-oss-120b") -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "⚠️ No relevant sections found in the document."
    
    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""
    
    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [37]:
# ── The complete Vectorless RAG function ─────────────────────────────────────

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\n📝 Answer:\n{answer}")
    
    return answer

In [44]:
# ── Run the full pipeline ────────────────────────────────────────────────────
answer = vectorless_rag(
    query="What are the evaluation results provided?",
    tree=pageindex_tree
)

🔍 Query: What are the evaluation results provided?

🧠 Reasoning: The query asks for the evaluation results. In the document tree, sections related to results are under the Experiments chapter. The '4.4. Quantitative result' node (0014) explicitly mentions evaluatin...
🎯 Retrieved node IDs: ['0014']
📄 Sections found: ['4.4. Quantitative result']

📝 Answer:
The quantitative evaluation reports the following results:

- **VisionDrone dataset** – DMNet beats the previous best (ClusDet [26]) by 1–2 AP points across three backbones and attains the highest score of **29.4 AP** with a ResNeXt‑101 backbone. Its AP₇₅ improves by roughly **4 points**, and APₛₘₐₗₗ gains **more than 2 points** over the baselines (4.4. Quantitative result, ?).

- **UVADT dataset** – DMNet again surpasses ClusDet, achieving a new state‑of‑the‑art **14.7 AP** and showing consistent accuracy gains on small, medium, and large objects (4.4. Quantitative result, ?).

- **Inference speed** – On a GTX 1080 Ti, DMNet processe

In [47]:
# ── Test with multiple queries ───────────────────────────────────────────────
test_queries = [
    "What is the proposed method?",
    "How does object detection improved by density map?",
    "What are the results?",
]

for q in test_queries:
    print()
    ans = vectorless_rag(q, pageindex_tree, verbose=False)
    print(f"Q: {q}")
    print(f"A: {ans[:300]}...")
    print("-" * 55)


Q: What is the proposed method?
A: The proposed method is **DMNet – a Density‑Map‑guided detection network** that works in three stages:  

1. **Density‑map generation** – a network is trained to predict a density map for each aerial image.  
2. **Image‑cropping** – a sliding window is run over the density map; the summed pixel inten...
-------------------------------------------------------

Q: How does object detection improved by density map?
A: Object detection is boosted by using the density map to (1) locate regions that contain many objects, (2) crop those regions so that the detector works on higher‑resolution, object‑dense patches, and (3) fuse the detections from the crops with those from the full image.

* The density map encodes ob...
-------------------------------------------------------

Q: What are the results?
A: The experiments show that the proposed DMNet achieves the best detection performance on both benchmark datasets and that each component of the system contrib